# Final IMDN GPU Evaluation — x2, x3, and x4

This notebook evaluates the verified official IMDN checkpoints on Set5, Set14, BSD100, and Urban100 at all three required scales. It saves one atomic CSV checkpoint per dataset-scale group after every image. If Colab disconnects, rerun the notebook: completed images are validated and skipped.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Select a GPU runtime and reconnect.')
DEVICE = torch.device('cuda')
DATASETS = ('Set5', 'Set14', 'BSD100', 'Urban100')
SCALES = (2, 3, 4)
WARMUP_RUNS = 3
TIMED_RUNS = 10
SAVE_RECONSTRUCTIONS = True

print('GPU:', torch.cuda.get_device_name(DEVICE))
print('PyTorch:', torch.__version__)
print('Warm-ups/timed runs per image:', WARMUP_RUNS, '/', TIMED_RUNS)

In [ ]:
from app.deep_learning.checkpoints import (
    IMDN_CHECKPOINTS,
    download_official_imdn_checkpoint,
)

DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
CHECKPOINT_ROOT = DATA_ROOT / 'checkpoints'
OUTPUT_ROOT = DATA_ROOT / 'results' / 'phase3' / 'imdn' / 'full'
METRICS_ROOT = OUTPUT_ROOT / 'metrics'
SR_ROOT = OUTPUT_ROOT / 'reconstructions'
METRICS_ROOT.mkdir(parents=True, exist_ok=True)

checkpoint_paths = {
    scale: download_official_imdn_checkpoint(CHECKPOINT_ROOT, scale)
    for scale in SCALES
}
for scale in SCALES:
    print(f'IMDN x{scale} verified:', checkpoint_paths[scale])

In [ ]:
from app.evaluation.data_validation import validate_prepared_dataset

validations = {}
for dataset in DATASETS:
    for scale in SCALES:
        validation = validate_prepared_dataset(dataset, scale, DATA_ROOT)
        validations[(dataset, scale)] = validation
        print(f'VALID: {dataset} x{scale} — {validation.image_count} pairs')

expected_total = sum(
    validations[(dataset, scale)].image_count
    for dataset in DATASETS
    for scale in SCALES
)
if expected_total != 657:
    raise RuntimeError(f'Expected 657 dataset-scale image evaluations; found {expected_total}.')
print('Total expected image evaluations:', expected_total)

In [ ]:
from app.evaluation.experiment import read_results_csv

REQUIRED_RESUME_FIELDS = {
    'dataset', 'image', 'scale', 'method', 'checkpoint_sha256',
    'timing_device', 'timing_scope', 'psnr_y', 'ssim_y',
    'psnr_rgb', 'ssim_rgb', 'latency_mean_ms', 'latency_median_ms',
}

def validated_resume_records(csv_path, dataset, scale, expected_names):
    records = read_results_csv(csv_path)
    if not records:
        return []
    missing_fields = REQUIRED_RESUME_FIELDS - set(records[0])
    if missing_fields:
        raise ValueError(f'{csv_path} has an old/incomplete schema: {sorted(missing_fields)}')
    seen = set()
    for record in records:
        if record['dataset'] != dataset or record['scale'] != f'x{scale}':
            raise ValueError(f'{csv_path} contains the wrong dataset or scale.')
        if record['method'] != 'imdn' or record['timing_device'] != 'gpu':
            raise ValueError(f'{csv_path} is not an IMDN GPU result checkpoint.')
        if record['checkpoint_sha256'] != IMDN_CHECKPOINTS[scale].sha256:
            raise ValueError(f'{csv_path} used a different checkpoint.')
        if record['image'] not in expected_names:
            raise ValueError(f"Unexpected image in {csv_path}: {record['image']}")
        if record['image'] in seen:
            raise ValueError(f"Duplicate image in {csv_path}: {record['image']}")
        seen.add(record['image'])
    return records

In [ ]:
from app.deep_learning.imdn import load_pretrained_imdn
from app.evaluation.experiment import write_results_csv
from app.evaluation.images import pair_image_paths
from app.evaluation.imdn import IMDNEvaluationConfig, evaluate_imdn_image

all_records = []
completed_total = 0
for scale in SCALES:
    model = load_pretrained_imdn(checkpoint_paths[scale], scale, DEVICE)
    print(f'\nLoaded IMDN x{scale}')
    for dataset in DATASETS:
        validation = validations[(dataset, scale)]
        pairs = pair_image_paths(validation.hr_directory, validation.lr_directory)
        expected_names = {hr_path.name for hr_path, _ in pairs}
        metrics_path = METRICS_ROOT / f'{dataset}_x{scale}_imdn_gpu.csv'
        records = validated_resume_records(
            metrics_path, dataset, scale, expected_names
        )
        completed_names = {record['image'] for record in records}
        print(
            f'{dataset} x{scale}: resuming with '
            f'{len(completed_names)}/{len(pairs)} complete'
        )
        config = IMDNEvaluationConfig(
            dataset=dataset,
            scale=scale,
            warmup_runs=WARMUP_RUNS,
            timed_runs=TIMED_RUNS,
        )
        sr_output_dir = SR_ROOT / dataset / f'x{scale}' if SAVE_RECONSTRUCTIONS else None
        for index, (hr_path, lr_path) in enumerate(pairs, start=1):
            if hr_path.name in completed_names:
                continue
            record = evaluate_imdn_image(
                hr_path,
                lr_path,
                model,
                config,
                DEVICE,
                sr_output_dir=sr_output_dir,
            )
            if records and set(record) != set(records[0]):
                raise ValueError(f'Resume schema changed for {metrics_path}.')
            records.append(record)
            write_results_csv(records, metrics_path, overwrite=True)
            completed_names.add(hr_path.name)
            completed_total += 1
            print(
                f'  [{index:03d}/{len(pairs):03d}] {hr_path.name} — '
                f"PSNR-Y {record['psnr_y']:.3f}, "
                f"GPU {record['latency_mean_ms']:.2f} ms"
            )
        if len(records) != len(pairs):
            raise RuntimeError(
                f'{dataset} x{scale} ended with {len(records)}/{len(pairs)} records.'
            )
        all_records.extend(records)
        print(f'COMPLETE: {dataset} x{scale} -> {metrics_path}')
    del model
    torch.cuda.empty_cache()

if len(all_records) != expected_total:
    raise RuntimeError(f'Expected {expected_total} final records; found {len(all_records)}.')
print(f'\nAll {len(all_records)} IMDN image evaluations are complete.')

In [ ]:
from app.evaluation.reporting import summarize_deep_learning_results

dataset_order = {name: index for index, name in enumerate(DATASETS)}
summaries = summarize_deep_learning_results(all_records)
summaries.sort(
    key=lambda row: (dataset_order[row['dataset']], int(row['scale'][1:]))
)
all_metrics_path = METRICS_ROOT / 'imdn_all_gpu.csv'
summary_path = METRICS_ROOT / 'imdn_summary_gpu.csv'
write_results_csv(all_records, all_metrics_path, overwrite=True)
write_results_csv(summaries, summary_path, overwrite=True)

print('\nIMDN SUMMARY')
for row in summaries:
    print(
        f"{row['dataset']:8s} {row['scale']} | "
        f"PSNR-Y {row['psnr_y']:.3f} | SSIM-Y {row['ssim_y']:.4f} | "
        f"GPU {row['latency_mean_ms']:.2f} ms | "
        f"adjusted {row['dimension_adjusted_count']}/{row['image_count']}"
    )
print('\nCombined metrics:', all_metrics_path)
print('Summary:', summary_path)

## Completion test

The run is complete only when it reports **657 IMDN image evaluations**, prints 12 summary rows, and saves `imdn_all_gpu.csv` plus `imdn_summary_gpu.csv`. A disconnect is not a failure: reconnect to a GPU and run all cells again to resume from the per-image CSV checkpoints. Do not delete partial CSV files.